In [40]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [78]:
import yfinance as yf
import pandas as pd
import numpy as np
import reused as r
import scipy as sp

Housekeeping items (delete later)

implement:

Create a correlation matrix
Create a static covariance matrix
Make a 3 year vs 10-year covariance comparison.

In [79]:
# All following functions must be calculated using 
# log returns, as they use geometric means.

# Now implementng annualized geomean daily returns

def annualized_daily_returns(ret_df: pd.DataFrame) -> pd.Series:
    '''
    Finds the annualized daily returns over the entire time
    period of a ticker. This the mean daily returns are calculated 
    using the geometric average. 

    Parameters:
    ret_df: a DataFrame containing log returns of a specific 
    ticker.

    Returns:
    A Series, containing the annualized daily return.
    '''
    df = ret_df.dropna().astype(float)
    
    return np.exp(df.mean() * 252) - 1

# Implementing annualized geomean volatility.

def annualized_volatility(ret_df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    '''
    Finds the annualized daily volatility over the entire time
    period of a ticker. This is the geometric average of volatility
    over time.

    Parameters:
    ret_df: a DataFrame containing the log returns of a specific 
    ticker.

    Returns:
    The geometric average of the annualized volatility calculated from the rolling 
    volatility window, and a more direct way to calculate the volatility, using stdev.
    '''
    vol_df = r.volatility(ret_df, 30).dropna().astype(float)

    av = (np.exp(vol_df.dropna().mean() * np.sqrt(252)) - 1) * (ret_df.std() * np.sqrt(252))

    return av ** (1/2)

In [118]:
etf_list: list = ["VDC", "VCR", "DTCR", "USRT", "FNCL","VOO", "SOXX"]

In [122]:
portfolio = r.log_returns(r.load_stock_data(etf_list, "2021-01-01", "2026-01-01"))

[*********************100%***********************]  7 of 7 completed


In [123]:
portfolio.corr()

,DTCR,FNCL,SOXX,USRT,VCR,VDC,VOO
DTCR,1.000000,0.488431,0.655864,0.667782,0.610600,0.371748,0.693151
FNCL,0.488431,1.000000,0.550396,0.656093,0.709632,0.525466,0.799837
SOXX,0.655864,0.550396,1.000000,0.441306,0.725554,0.220035,0.815256
USRT,0.667782,0.656093,0.441306,1.000000,0.625449,0.601289,0.674666
VCR,0.610600,0.709632,0.725554,0.625449,1.000000,0.446518,0.878475
VDC,0.371748,0.525466,0.220035,0.601289,0.446518,1.000000,0.538276
VOO,0.693151,0.799837,0.815256,0.674666,0.878475,0.538276,1.000000


In [124]:
ret = annualized_daily_returns(portfolio)
print(ret)

DTCR    0.083885
FNCL    0.155507
SOXX    0.200970
USRT    0.070356
VCR     0.086796
VDC     0.067755
VOO     0.148035
dtype: float64


In [125]:
vol = annualized_volatility(portfolio)
print(vol)

DTCR    0.191084
FNCL    0.171047
SOXX    0.308454
USRT     0.15744
VCR     0.204413
VDC     0.104908
VOO     0.137985
dtype: object


In [126]:
annualized_cov = portfolio.cov() * 252

In [127]:
hysa_rfr = 0.034
ret_vol_df = pd.DataFrame(data={"Returns": ret, "Volatility": vol}, index = ret.index)

In [128]:
ret_vol_df["Sharpe"] = (ret_vol_df["Returns"] - hysa_rfr) / ret_vol_df["Volatility"]

In [129]:
ret_vol_df.head(7)

,Returns,Volatility,Sharpe
DTCR,0.083885,0.191084,0.261062
FNCL,0.155507,0.171047,0.710375
SOXX,0.200970,0.308454,0.541311
USRT,0.070356,0.15744,0.230919
VCR,0.086796,0.204413,0.258279
VDC,0.067755,0.104908,0.321756
VOO,0.148035,0.137985,0.82643


Now it is time for optimization with scipy.

In [157]:
returns = ret.to_numpy()              # your annualized returns Series -> array
cov_matrix = annualized_cov.to_numpy() # your annualized covariance matrix -> array
rf = 0.034
n = len(returns)

def neg_sharpe(weights):
    port_return = weights @ returns
    port_vol = np.sqrt(weights @ cov_matrix @ weights)
    return -(port_return - rf) / port_vol

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = tuple((0.03, 0.3) for _ in range(n))
initial_guess = np.array([1/n] * n)

result = sp.optimize.minimize(neg_sharpe, initial_guess, method='SLSQP',
                   bounds=bounds, constraints=constraints)

optimal_weights = result.x

for ticker, weight in zip(ret.index, optimal_weights):
    print(f"{ticker}: {weight:.4f}")

DTCR: 0.0300
FNCL: 0.3000
SOXX: 0.1491
USRT: 0.0300
VCR: 0.0300
VDC: 0.1609
VOO: 0.3000
